In [2]:
import pandas as pd
import numpy as np
import re 

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import sent_tokenize

from nltk import sent_tokenize

from gensim.models import Word2Vec
import nltk

nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [5]:
messages = pd.read_csv(
    'SMSSpamCollection.txt', sep='\t', names=['label', 'message']
)

messages.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
# Text Preprocessing 
lemmatizer = WordNetLemmatizer()
corpus=[]
for i in range(len(messages)):
    review = re.sub('[^a-zA-Z]', ' ', messages['message'][i])
    review = review.lower()
    review = review.split()

    review=[
        lemmatizer.lemmatize(word)
        for word in review
        if word not in stopwords.words('english')
    ]

    review = ' '.join(review)

    corpus.append(review)

In [8]:
# word2vec /
words =[] 
for sentence in corpus:
    words.append(sentence.split())
    


In [12]:
words[:5]

[['go',
  'jurong',
  'point',
  'crazy',
  'available',
  'bugis',
  'n',
  'great',
  'world',
  'la',
  'e',
  'buffet',
  'cine',
  'got',
  'amore',
  'wat'],
 ['ok', 'lar', 'joking', 'wif', 'u', 'oni'],
 ['free',
  'entry',
  'wkly',
  'comp',
  'win',
  'fa',
  'cup',
  'final',
  'tkts',
  'st',
  'may',
  'text',
  'fa',
  'receive',
  'entry',
  'question',
  'std',
  'txt',
  'rate',
  'c',
  'apply'],
 ['u', 'dun', 'say', 'early', 'hor', 'u', 'c', 'already', 'say'],
 ['nah', 'think', 'go', 'usf', 'life', 'around', 'though']]

In [13]:
##train word2vec 

model = Word2Vec(words, vector_size=100, window=5, min_count=1, workers=4)

In [15]:
# test word2vec 
# model.wv['love']
print(model.wv['love'].shape)



(100,)


In [24]:
####################v--------AvGWord2Vec ---------##############

def avg_word2vec(doc):
    vectors=[
        model.wv[word]
        for word in doc
        if word in model.wv.index_to_key
    ]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    
    return np.mean(vectors, axis=0)


In [25]:
##create features 

X=[]
for doc in words:
    X.append(avg_word2vec(doc))

X = np.array(X)

print(X.shape)

(5572, 100)


In [26]:
##encode target 
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(messages['label'])

print(y.shape)

(5572,)


In [27]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size=0.20, random_state=42
)

In [28]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier(n_estimators=100, random_state=42)

classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)

##print accuracy score 

from sklearn.metrics import accuracy_score
print(accuracy_score(y_test, y_pred))


0.9605381165919282
